# EarnPrem Pitch EDA: do stocks rise into scheduled earnings announcements?

**The anomaly.** Stocks earn abnormal returns around scheduled earnings announcements (Frazzini & Lamont 2007; Savor
& Wilson 2016): attention-driven buying beforehand, and a premium for announcement risk.

**Design.** Current S&P 500 members (survivorship), earnings dates = 8-K Item 2.02 filing dates (`Buyback Pitch`).
To avoid needing the date in advance, the *predicted* date = the same quarter's announcement a year earlier + 364
days (kept only when the actual date lands within 3 weeks). Windows around the predicted date P: P−10…P−2 (run-up),
P−1…P+1 (announcement), P+2…P+11. Returns vs the equal-weighted universe, averaged per week. 2006–15 vs 2016–.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
e=pd.read_csv("../Buyback Pitch/data/earnings.csv",parse_dates=["date"]).sort_values(["ticker","date"])
e=e[e.groupby("ticker").date.diff().dt.days.fillna(999)>45]
p=pd.read_csv("../Overnight Pitch/data/panel.csv",parse_dates=["date"])
A=p.pivot(index="date",columns="ticker",values="adj_close").drop(columns="SPY"); A=A.where(A>0)
RD=np.log(A).diff(); RD=RD.where(RD.abs()<0.5); X=RD.sub(RD.mean(1),axis=0)*1e4; cal=X.index
col={t:j for j,t in enumerate(X.columns)}; Xv=X.values
rows=[]
for t,g in e.groupby("ticker"):
    if t not in col: continue
    d=g.date.values
    for k in range(1,len(d)):
        prev=pd.Timestamp(d[k-1]); cur=pd.Timestamp(d[k])
        # predicted date = previous same-quarter announcement + ~91 days? use one-year-ago announcement + 364 days (known a year ahead)
        if k<4: continue
        yr=pd.Timestamp(d[k-4])
        pred=yr+pd.Timedelta(days=364)
        if abs((cur-pred).days)>21: continue
        i=cal.searchsorted(pred)                                   # predicted announcement day (known in advance)
        if i<15 or i+15>=len(cal): continue
        x=Xv[:,col[t]]
        rows.append({"t":t,"date":cal[i],"pre10":np.nansum(x[i-10:i-1]),"win":np.nansum(x[i-1:i+2]),"post10":np.nansum(x[i+2:i+12])})
E=pd.DataFrame(rows); E["era"]=np.where(E.date<"2016-01-01","2006-15","2016-")
for era in ["2006-15","2016-"]:
    d=E[E.era==era].assign(w=lambda z:z.date.dt.to_period("W"))
    g=d.groupby("w")[["pre10","win","post10"]].mean()
    print(era,"events",len(d),{c:f"{g[c].mean():+.1f}bp (t {g[c].mean()/g[c].std()*np.sqrt(len(g)):+.2f}, weeks {len(g)})" for c in g})

2006-15 events 15002 {'pre10': '+27.9bp (t +3.12, weeks 525)', 'win': '+26.9bp (t +2.90, weeks 525)', 'post10': '+3.8bp (t +0.37, weeks 525)'}
2016- events 19078 {'pre10': '+17.4bp (t +1.97, weeks 548)', 'win': '+1.7bp (t +0.19, weeks 548)', 'post10': '+4.0bp (t +0.38, weeks 548)'}


**Interpretation.**
- **The run-up is still there but smaller:** +28bp (t 3.1) in 2006–15, +17bp (t 2.0) since 2016.
- **The announcement-window premium has gone** (+27bp, t 2.9 → +2bp, t 0.2).
- **Too small to trade.** Run-up gross Sharpe is about 0.6 a year; after a 10bp large-cap round trip the edge is about
  7bp per event (Sharpe ≈ 0.25). Survivorship flatters it.

**Verdict: real but not tradable after costs.**